# 07_rag_baseline.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook implementa el primer **RAG baseline funcional** del proyecto.

## Arquitectura implementada

```text
chunks_recursive.parquet
        ↓
Embeddings BGE-M3
        ↓
Retriever Top-K por similitud coseno
        ↓
Construcción de contexto
        ↓
LLM
        ↓
Respuesta con evidencia documental
```

## Objetivo

Permitir consultas sobre documentos de aseguramiento técnico e interventoría, recuperando evidencia documental trazable.

## Entradas
- `chunks_recursive.parquet`
- Opcional: `gold_questions.csv`

## Salidas
- `rag_baseline_results.csv`
- `rag_baseline_results.xlsx`
- `rag_contexts.csv`


In [ ]:
# =====================================================
# 07_rag_baseline.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas pyarrow sentence-transformers scikit-learn openpyxl openai

## 1. Importar librerías

In [ ]:
import os
import textwrap
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 160)

print('Librerías cargadas correctamente')

## 2. Cargar archivos

Sube:
- `chunks_recursive.parquet`
- Opcionalmente `gold_questions.csv` para probar preguntas del gold standard.

In [ ]:
print('Sube chunks_recursive.parquet y, opcionalmente, gold_questions.csv')
uploaded = files.upload()
print('\nArchivos cargados:', list(uploaded.keys()))

## 3. Leer chunks y preguntas

In [ ]:
chunks_path = Path('chunks_recursive.parquet')

if not chunks_path.exists():
    raise FileNotFoundError('No se encontró chunks_recursive.parquet')

chunks_df = pd.read_parquet(chunks_path).reset_index(drop=True)

required_cols = {'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id', 'chunk_text'}
missing = required_cols - set(chunks_df.columns)

if missing:
    raise ValueError(f'Faltan columnas en chunks_recursive.parquet: {missing}')

print('Chunks cargados:', len(chunks_df))
display(chunks_df.head(3))

gold_questions = None
if Path('gold_questions.csv').exists():
    gold_questions = pd.read_csv('gold_questions.csv')
    print('\nPreguntas gold cargadas:', len(gold_questions))
    display(gold_questions.head(3))

## 4. Cargar modelo de embeddings ganador

Según la evaluación manual del Notebook 06, se seleccionó `BAAI/bge-m3` como modelo de embeddings para el RAG baseline.

In [ ]:
EMBEDDING_MODEL_NAME = 'BAAI/bge-m3'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print('Modelo cargado:', EMBEDDING_MODEL_NAME)

## 5. Generar embeddings del corpus

In [ ]:
texts = chunks_df['chunk_text'].astype(str).tolist()

chunk_embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print('Embeddings generados')
print('Shape:', chunk_embeddings.shape)

## 6. Implementar retriever Top-K

El retriever convierte la pregunta en embedding y recupera los chunks más similares por similitud coseno.

In [ ]:
def retrieve_context(question, top_k=5):
    """Recupera los top-k chunks más relevantes para una pregunta."""
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(question_embedding, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        results.append({
            'rank': rank,
            'score': float(scores[idx]),
            'doc_id': row['doc_id'],
            'filename': row['filename'],
            'tipo_documento': row['tipo_documento'],
            'page': row['page'],
            'chunk_id': row['chunk_id'],
            'chunk_text': row['chunk_text']
        })

    return pd.DataFrame(results)

## 7. Probar recuperación documental

In [ ]:
pregunta_prueba = '¿Qué incumplimientos contractuales fueron identificados en los documentos?'

contexto_prueba = retrieve_context(pregunta_prueba, top_k=5)

display(contexto_prueba[['rank', 'score', 'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id']])

for _, row in contexto_prueba.iterrows():
    print('='*100)
    print(f"Rank {row['rank']} | Score {row['score']:.4f} | {row['filename']} | {row['chunk_id']}")
    print('-'*100)
    print(row['chunk_text'][:1000])
    print()

## 8. Configurar LLM

Este notebook está preparado para usar OpenAI.  
Debes ingresar tu API key en Colab cuando se solicite.

Si no quieres llamar al LLM todavía, puedes saltar esta sección y revisar solo la recuperación documental.

In [ ]:
from getpass import getpass

USE_LLM = True

if USE_LLM:
    api_key = getpass('Ingrese OPENAI_API_KEY: ')
    os.environ['OPENAI_API_KEY'] = api_key
    print('API key configurada en la sesión')
else:
    print('LLM desactivado. Solo se ejecutará retrieval.')

## 9. Funciones para construir prompt y llamar al LLM

In [ ]:
def construir_contexto(context_df, max_chars_per_chunk=1200):
    """Construye contexto textual con citas trazables."""
    bloques = []

    for _, row in context_df.iterrows():
        texto = str(row['chunk_text'])[:max_chars_per_chunk]

        bloque = f"""
[Fuente {row['rank']}]
doc_id: {row['doc_id']}
archivo: {row['filename']}
tipo_documento: {row['tipo_documento']}
pagina: {row['page']}
chunk_id: {row['chunk_id']}
texto: {texto}
"""
        bloques.append(bloque.strip())

    return '\n\n'.join(bloques)


def construir_prompt(question, context_df):
    contexto = construir_contexto(context_df)

    prompt = f"""
Eres un asistente experto en aseguramiento técnico, interventoría y análisis de riesgos documentales.

Tu tarea es responder la pregunta usando únicamente el contexto documental entregado.

Reglas:
1. No inventes información.
2. Si el contexto no contiene evidencia suficiente, dilo explícitamente.
3. Responde en español.
4. Usa lenguaje claro y profesional.
5. Incluye siempre evidencia documental indicando doc_id, archivo, página y chunk_id.
6. Si identificas riesgos, organízalos en bullets.

Pregunta:
{question}

Contexto documental recuperado:
{contexto}

Formato de respuesta esperado:

Respuesta:
[respuesta sintetizada]

Evidencia utilizada:
- doc_id | archivo | página | chunk_id | breve justificación

Limitaciones:
[indica si falta información o si la evidencia es parcial]
"""
    return prompt.strip()


def llamar_llm(prompt, model='gpt-4o-mini'):
    """Llama a OpenAI para generar una respuesta RAG."""
    from openai import OpenAI

    client = OpenAI()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'Responde siempre con base en evidencia documental. No inventes datos.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.0
    )

    return response.choices[0].message.content


def rag_answer(question, top_k=5, llm_model='gpt-4o-mini'):
    """Ejecuta el flujo completo RAG: retrieve + prompt + LLM."""
    context_df = retrieve_context(question, top_k=top_k)
    prompt = construir_prompt(question, context_df)

    if USE_LLM:
        answer = llamar_llm(prompt, model=llm_model)
    else:
        answer = 'LLM desactivado. Se retornan únicamente los chunks recuperados.'

    return answer, context_df, prompt

## 10. Ejecutar primer RAG baseline

In [ ]:
question = '¿Qué incumplimientos contractuales fueron identificados en los documentos?'

answer, context_df, prompt = rag_answer(question, top_k=5)

print('PREGUNTA:')
print(question)
print('\nRESPUESTA RAG:')
print(answer)

print('\nFUENTES RECUPERADAS:')
display(context_df[['rank', 'score', 'doc_id', 'filename', 'page', 'chunk_id']])

## 11. Ejecutar RAG sobre preguntas del gold standard

Para controlar costos, se ejecuta inicialmente sobre 5 preguntas prioritarias. Puedes aumentar el número después.

In [ ]:
rag_results = []
rag_contexts = []

if gold_questions is not None:
    preguntas_prioritarias = ['Q001', 'Q002', 'Q003', 'Q004', 'Q020']
    questions_to_run = gold_questions[gold_questions['question_id'].isin(preguntas_prioritarias)].copy()
else:
    questions_to_run = pd.DataFrame([
        {'question_id': 'MANUAL_001', 'question': question, 'categoria': 'manual'}
    ])

for _, qrow in questions_to_run.iterrows():
    qid = qrow['question_id']
    qtext = qrow['question']
    categoria = qrow.get('categoria', '')

    print('='*100)
    print('Ejecutando:', qid, qtext)

    answer, context_df, prompt = rag_answer(qtext, top_k=5)

    rag_results.append({
        'question_id': qid,
        'question': qtext,
        'categoria': categoria,
        'rag_answer': answer,
        'top_k': 5,
        'embedding_model': EMBEDDING_MODEL_NAME,
        'llm_used': USE_LLM
    })

    temp_context = context_df.copy()
    temp_context['question_id'] = qid
    temp_context['question'] = qtext
    temp_context['categoria'] = categoria
    rag_contexts.append(temp_context)

rag_results_df = pd.DataFrame(rag_results)
rag_contexts_df = pd.concat(rag_contexts, ignore_index=True)

display(rag_results_df)
display(rag_contexts_df[['question_id', 'rank', 'score', 'doc_id', 'filename', 'page', 'chunk_id']].head(20))

## 12. Crear plantilla de evaluación del RAG

In [ ]:
rag_evaluation_template = rag_results_df.copy()
rag_evaluation_template['respuesta_correcta_manual'] = ''
rag_evaluation_template['usa_evidencia_manual'] = ''
rag_evaluation_template['alucina_manual'] = ''
rag_evaluation_template['comentario_evaluador'] = ''

display(rag_evaluation_template)

## 13. Guardar resultados

In [ ]:
rag_results_df.to_csv('rag_baseline_results.csv', index=False, encoding='utf-8-sig')
rag_results_df.to_excel('rag_baseline_results.xlsx', index=False)
rag_contexts_df.to_csv('rag_contexts.csv', index=False, encoding='utf-8-sig')
rag_evaluation_template.to_excel('rag_evaluation_template.xlsx', index=False)

print('Archivos generados:')
print('- rag_baseline_results.csv')
print('- rag_baseline_results.xlsx')
print('- rag_contexts.csv')
print('- rag_evaluation_template.xlsx')

## 14. Descargar resultados

In [ ]:
files.download('rag_baseline_results.csv')
files.download('rag_baseline_results.xlsx')
files.download('rag_contexts.csv')
files.download('rag_evaluation_template.xlsx')